# Home energy flexibility: measured savings, modelled payback, and who captures the value

**Claim cluster:** `C043`, `C044`, `C045`, `C046`, `C047`, `C048`, `C049`, `C050`, `C055`, `C056`, `C057`, `C058`, `C061`  
**Source of truth:** `data/smarthome.duckdb` (read-only), via `src/viz_core.py`

In [ ]:
import os
import sys
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, clear_output, display

# Find the repo root by walking up until schemas/stage2.sql appears, so the
# notebook runs from notebooks/, from the repo root, or from anywhere else.
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "schemas" / "stage2.sql").exists():
        ROOT = _candidate
        break
else:  # pragma: no cover - only reachable outside a checkout
    raise FileNotFoundError("schemas/stage2.sql not found above " + str(_here))

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

# Option A topology: this wave reads one dedicated Stage 2 database. Point
# viz_core at it before import -- it resolves the path once at import time --
# so every fetch in this kernel uses the same file.
os.environ["SMARTHOME_DB"] = str(ROOT / "data" / "smarthome.duckdb")

import viz_core as viz  # noqa: E402

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

# Renderers matter for two reasons. "browser" (plotly's default) opens a
# window when a figure is shown outside a live kernel, which breaks headless
# execution. The bare "notebook" renderer in plotly 7 emits only text/html,
# carrying no figure payload, so a notebook could run clean and still show
# nothing. "plotly_mimetype+notebook" emits the figure JSON for JupyterLab and
# an inline HTML fallback elsewhere -- which is also what the headless test
# asserts on.
pio.renderers.default = "plotly_mimetype+notebook"

# Every figure in this notebook reads only from data/smarthome.duckdb (read-only).
CLAIM_IDS = ('C043', 'C044', 'C045', 'C046', 'C047', 'C048', 'C049', 'C050', 'C055', 'C056', 'C057', 'C058', 'C061')
assert viz.DEFAULT_DB_PATH.endswith("smarthome.duckdb"), viz.DEFAULT_DB_PATH

claims = viz.fetch_claims(list(CLAIM_IDS))
sources = viz.sources_for_claims(list(CLAIM_IDS)) if CLAIM_IDS else pd.DataFrame()
cluster_metrics = (
    viz.claim_metrics_lookup(list(CLAIM_IDS)) if CLAIM_IDS else pd.DataFrame()
)

# Corpus-wide row counts, read once so a notebook can state the size of what it
# is drawing from without confusing a whole-corpus count with the size of its
# own (possibly empty) claim cluster.
CORPUS = viz.connect_db()
try:
    corpus_counts = {
        table: CORPUS.execute(f"SELECT COUNT(1) FROM {table}").fetchone()[0]
        for table in ("stage2_claim", "metric", "entity", "source_mirror",
                      "predicate", "extraction_decision")
    }
finally:
    CORPUS.close()

def provenance(claim_id):
    """Collapsible evidence panel for one claim in this cluster."""
    panel = viz.render_provenance_panel(claim_id)
    return widgets.VBox([panel])

def show(*items):
    """Display data frames, HTML, or figures, one per argument."""
    for item in items:
        display(item)

def confidence_legend():
    """Show the colour coding applied to confidence across all figures."""
    display(HTML(
        "<b>Confidence colour coding:</b> "
        f"<span style='color:{viz.CONFIDENCE_COLORS['high']}'>high</span> &middot; "
        f"<span style='color:{viz.CONFIDENCE_COLORS['medium']}'>medium</span> &middot; "
        f"<span style='color:{viz.CONFIDENCE_COLORS['low']}'>low</span>"
    ))

## Provenance

The energy cluster mixes measured field results with modelled scenario assumptions, so each claim's evidence class matters more here than elsewhere. The panels below make the split visible before any number is charted.

In [ ]:
for claim_id in CLAIM_IDS:
    display(HTML(f"<h4>{claim_id} &mdash; {claims.set_index('local_id').loc[claim_id, 'claim_text']}</h4>"))
    display(provenance(claim_id))
confidence_legend()

## Primary evidence

### Flexibility by device, as the corpus records it

Each row is a metric from the cluster, parsed from its authored value cell. Range metrics (`29-54`, `15-25`) are drawn as a bar spanning the stated range rather than collapsed to a midpoint, because the corpus never states a single value for them. Bars are coloured by the metric's recorded confidence, not by the claim's.

In [ ]:
# Flex metrics for this cluster: pull the exact metric ids we plot, then read
# their values straight from the database. range_frame turns each authored
# value cell into (low, high) bounds and keeps the raw cell for the hover.
FLEX_METRIC_IDS = [
    "M089", "M094", "M095", "M096", "M097", "M099", "M100", "M103", "M104",
    "M105",
]
flex = viz.range_frame(FLEX_METRIC_IDS)
show(flex[["metric", "name", "value", "low", "high", "unit", "confidence",
           "source"]])

_records = flex.to_dict("records")
fig = go.Figure(go.Bar(
    x=[(r["low"] + r["high"]) / 2 for r in _records],
    y=list(reversed(flex["metric"] + " " + flex["name"])),
    orientation="h",
    marker=dict(color=[viz.CONFIDENCE_COLORS.get(c, viz.CONFIDENCE_COLORS[None])
                       for c in flex["confidence"]]),
    error_x=dict(
        type="data",
        array=[abs(r["high"] - r["low"]) / 2 for r in _records],
        arrayminus=[abs(r["high"] - r["low"]) / 2 for r in _records],
        visible=True,
    ),
    customdata=list(zip(flex["value"], flex["unit"], flex["source"],
                        flex["confidence"])),
    hovertemplate=(
        "%{y}<br>stated value: %{customdata[0]} %{customdata[1]}"
        "<br>source: %{customdata[2]}<br>confidence: %{customdata[3]}"
        "<extra></extra>"
    ),
))
fig.update_layout(
    title="Stated flexibility magnitudes (range metrics drawn as ranges)",
    xaxis_title="percent, or kW where the unit says so",
    height=430,
    yaxis=dict(automargin=True),
)
fig.show()

## Supporting context

### Payback and value distribution are modelled, not measured

`C061` and `C055` compare modelled payback across five systems; `C057` reports the modelled split of the value that results. Both are recorded as `inference` at medium confidence because the extraction log decision `L034` classifies the whole model as scenario assumptions rather than measurements. The two charts below therefore sit in a separate band from the field measurements above, and are labelled as modelled rather than presented as results.

In [ ]:
MODELLED_SECTIONS = [
    "Payback and NPV", "5. Value Distribution", "6. Sensitivity Analysis",
]
modelled = pd.concat(
    [viz.section_metrics(section) for section in MODELLED_SECTIONS],
    ignore_index=True,
)
show(modelled[["local_id", "metric_name", "value", "unit", "confidence",
               "section"]].rename(
    columns={"local_id": "metric", "metric_name": "name",
             "confidence": "conf", "source_ref": "source"}))

# Payback: M131-M134 are single-year figures, M135 carries a full system
# result including its own NPV. Read them by id rather than by section filter.
PAYBACK_IDS = ["M132", "M133", "M134", "M131", "M135"]
payback = viz.metrics_by_id(PAYBACK_IDS)
payback["years"] = [viz.range_value(v)[0] for v in payback["value"]]
payback = payback.dropna(subset=["years"]).sort_values("years")
display(HTML("<h4>Modelled payback horizon</h4>"))
show(payback[["local_id", "metric_name", "value", "years", "confidence"]]
     .rename(columns={"local_id": "metric", "metric_name": "name",
                      "confidence": "conf"}))

fig = go.Figure(go.Bar(
    x=payback["years"],
    y=payback["local_id"] + " " + payback["metric_name"],
    orientation="h",
    marker=dict(color=[viz.CONFIDENCE_COLORS.get(c, viz.CONFIDENCE_COLORS[None])
                       for c in payback["confidence"]]),
    customdata=list(zip(payback["value"], payback["confidence"])),
    hovertemplate="%{y}<br>stated: %{customdata[0]}<br>confidence: %{customdata[1]}"
                  "<extra></extra>",
))
fig.update_layout(
    title="Modelled payback, years (scenario assumptions, not measurements)",
    xaxis_title="years", height=300, yaxis=dict(automargin=True),
    annotations=[dict(text="L034: model values recorded as inference at "
                           "medium confidence", xref="paper", yref="paper",
                       x=0, y=-0.25, showarrow=False, font=dict(size=11))],
)
fig.show()

# Value distribution: the corpus states a share and an absolute annual figure
# in the same cell, sometimes as a range. Both come from the same value cell.
VALUE_IDS = ["M136", "M137", "M138", "M139", "M140"]
_raw_value = viz.metrics_by_id(VALUE_IDS)
value_split = pd.DataFrame([
    {
        "metric": record["local_id"],
        "name": record["metric_name"],
        "stated": record["value"],
        "share_pct": (nums[-1] if (nums := viz.parse_numbers(record["value"]))
                      else None),
        "confidence": record["confidence"],
    }
    for record in _raw_value.to_dict("records")
])
display(HTML("<h4>Modelled value distribution</h4>"))
show(value_split)

_share_total = value_split["share_pct"].sum()
display(HTML(
    f"<p>The five recorded shares total <b>{_share_total:g}%</b> of modelled "
    "annual value. The corpus records no row for the remainder, so the pie is "
    "normalised by Plotly and the missing share is not labelled as a "
    "recipient.</p>"
))

fig = go.Figure(go.Pie(
    labels=[f"{r['metric']} {r['name']}" for r in value_split.to_dict("records")],
    values=[r["share_pct"] or 0 for r in value_split.to_dict("records")],
    customdata=[r["stated"] for r in value_split.to_dict("records")],
    hovertemplate="%{label}<br>stated: %{customdata}<extra></extra>",
    hole=0.35,
))
fig.update_layout(title="Modelled annual flexibility value by recipient (%)",
                  height=380)
fig.show()

## Uncertainty and gaps

### The override-rate conflict is unresolved and shown, not averaged

Extraction-log decision `L019` records that the same UK heat-pump trial is reported with an override rate of `2.7%` in one note and `1.1-1.3%` (rising to `4.4%` for two-hour events) in another. Both are retained as separate metric rows; the protocol forbids discarding one, and forbids averaging them. The figure below prints them side by side, which is the honest presentation of a conflict the corpus could not resolve.

Other limits on this page:

- **Modelled and measured are not mixed.** Payback and value distribution come from a scenario model (`L034`); field results come from named trials. They are charted in separate bands.
- **No single household baseline is presented as measured.** The baseline consumption metrics are model inputs, not survey data.
- **Sensitivity inputs are ranges, not distributions.** `M141`-`M144` give base values and ranges; there is no basis for a probability distribution in the corpus, so no tornado chart is drawn here.

In [ ]:
# The conflict is located by name rather than by remembering metric ids, so a
# re-import that renumbers M101/M102 still surfaces the conflict.
_conflict = pd.concat(
    [viz.section_metrics(section)
     for section in ("2. Device-Level Flexibility Modeling",
                     "What Savings Are Measured Rather Than Claimed?")],
    ignore_index=True,
)
_conflict = _conflict[_conflict["metric_name"].str.contains(
    "override", case=False, na=False)]
display(HTML("<h4>Unresolved conflict: heat-pump override rate</h4>"
             "<p>Both values are retained; the corpus explicitly refuses to "
             "merge them (decision <code>L019</code>). Neither is averaged.</p>"))
show(_conflict[["local_id", "metric_name", "value", "unit", "confidence",
                "source_ref", "scope_conditions"]].rename(
    columns={"local_id": "metric", "metric_name": "name",
             "source_ref": "source"}))

display(HTML("<h4>Sensitivity inputs as recorded (base and range, no "
             "distribution)</h4>"))
show(viz.section_metrics("6. Sensitivity Analysis")[
    ["local_id", "metric_name", "value", "unit", "confidence"]]
     .rename(columns={"local_id": "metric", "metric_name": "name",
                      "confidence": "conf"}))

display(HTML("<h4>Claims in this cluster with no falsifier</h4>"
             "<p>A missing falsifier means the corpus recorded no observation "
             "that would disconfirm the claim. The panel above flags these "
             "claims in <code>no_falsifier</code>.</p>"))
show(claims.loc[claims["falsifier_missing"], ["local_id", "claim_type",
                                              "confidence"]]
     .rename(columns={"local_id": "claim", "claim_type": "type",
                      "confidence": "conf"}))

display(HTML("<h4>Metrics with conditions not stated in the source</h4>"))
_gaps = cluster_metrics.loc[~cluster_metrics["conditions_stated"].fillna(False)]
if _gaps.empty:
    display(HTML("<p>No unstated conditions among the cluster's metrics.</p>"))
else:
    show(_gaps[["local_id", "metric_name", "value", "unit"]].rename(
        columns={"local_id": "metric", "metric_name": "name"}))

display(HTML("<h4>Extraction decisions that changed or qualified this material</h4>"))
_decisions = pd.DataFrame(
    viz.connect_db().execute(
        "SELECT local_id, decision_type, description, resolution "
        "FROM extraction_decision "
        "WHERE decision_type IN ('evidence_downgrade', 'classification_conflict', "
        "                           'falsifier_absent', 'condition_absent', "
        "                           'boundary_absent', 'scope_boundary', "
        "                           'omission') "
        "ORDER BY local_id"
    ).fetchall(),
    columns=["decision", "type", "description", "resolution"],
)
show(_decisions)

## Reproducibility

Every figure reads `metric` rows by local id from `data/smarthome.duckdb`. Range parsing goes through `viz.range_value` and `viz.parse_numbers`, both unit-tested in `src/tests/test_viz_core.py`, because the value cells are authored strings rather than typed numbers.

In [ ]:
# The exact queries behind every figure above, re-run verbatim.
import duckdb
con = duckdb.connect(str(ROOT / 'data' / 'smarthome.duckdb'), read_only=True)
print('database:', con.execute('SELECT current_database()').fetchone()[0])
for table in ('stage2_claim', 'metric', 'entity', 'source_mirror',
               'predicate', 'workflow_stage', 'extraction_decision'):
    total = con.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f'{table:22} {total:>6} rows')
con.close()